# 특허 기술이전 수요예측 — Colab GPU 실행 (full_pool 포함, resume 가능)

무료 **T4 GPU**로 실행. **중단돼도 같은 셀을 다시 실행하면 죽은 시드부터 이어서** 진행됩니다.

## ⚠️ 가장 먼저 (필수)
상단 **런타임 ▸ 런타임 유형 변경 ▸ T4 GPU ▸ 저장**. 그다음 셀을 **순서대로** 실행. (끊기면 GPU 설정 후 **셀 1** → 죽었던 셀 다시)

## 셀 1 — 셋업 (GPU + 최신코드 + 라이브러리 + 드라이브)
몇 번 돌려도 안전. `git pull`로 최신 코드(resume · full_pool)를 받습니다. **`DATA_DIR`를 본인이 업로드한 폴더 경로로 맞추세요.**

In [ ]:
import torch, os
assert torch.cuda.is_available(), '❌ GPU 없음! 런타임▸런타임 유형 변경▸T4 GPU 설정 후 다시 실행.'
if not os.path.exists('/content/repo'):
    !git clone https://github.com/joshlee614/Patent-Citation-Graph-Based-Technology-Transfer-Demand-Prediction-.git /content/repo
%cd /content/repo
!git pull -q
!pip install -q torch_geometric statsmodels
from google.colab import drive; drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/kipris_data'   # ← 업로드한 폴더 경로로 수정
EMB = os.path.join(DATA_DIR, 'patent_embeddings.pt')
import torch_geometric
print('files in DATA_DIR:', sorted(os.listdir(DATA_DIR)) if os.path.exists(DATA_DIR) else '❌ 경로 없음')
print('✅ GPU:', torch.cuda.get_device_name(0), '| EMB OK:', os.path.exists(EMB))
!git log --oneline -1

## 셀 1b — (선택) 완전 초기화: 이전 체크포인트 / 결과 폴더 삭제
**처음부터 깨끗이** 다시 돌리고 싶을 때만 실행. 업로드한 **데이터(csv/pt)는 건드리지 않고** 결과·resume 캐시 폴더만 지웁니다.

In [ ]:
import shutil, os
for d in ['ipm_results_final', 'ipm_results_random', 'ipm_check_s3', 'ipm_fp_temporal']:
    p = os.path.join(DATA_DIR, d)
    if os.path.exists(p):
        shutil.rmtree(p); print('🗑️  삭제:', p)
    else:
        print('— 없음(건너뜀):', p)
print('✅ 데이터 파일은 그대로:', sorted(f for f in os.listdir(DATA_DIR) if f.endswith(('.csv', '.pt'))))

## 셀 2 — 최종 temporal 10-seed + full_pool (~80분, **메인 결과**)
현실 프로토콜(시간분할 + 동일-IPC 하드네거티브) + `--nneg_sweep`(후보군 크기 민감도) + `--full_pool`(비샘플 전체-풀 §12). **끊기면 이 셀 그대로 다시 → 죽은 시드부터.**

> quota가 빠듯해 **시드 0에서 자꾸 죽으면**, 이 셀에서 `--nneg_sweep --full_pool`을 빼서 가볍게 먼저 끝내고(시드 0이 ~6분), full_pool은 아래 **셀 2b**로 따로 돌리세요.

In [ ]:
!python run_ipm_experiment.py --mode full --device cuda \
  --data_dir "{DATA_DIR}" --emb_path "{EMB}" --demand_sample 200 \
  --split temporal --nneg_sweep --full_pool \
  --artifact_dir /content/drive/MyDrive/kipris_data/ipm_results_final

## 셀 2b — (선택) full_pool + nneg_sweep만 따로 (1 seed, ~25분)
메인을 가볍게 돌렸을 때만 사용. §11(n_neg)·§12(full_pool) 표가 여기 md에 따로 생성됩니다.

In [ ]:
!python run_ipm_experiment.py --mode full --device cuda --seeds 1 \
  --data_dir "{DATA_DIR}" --emb_path "{EMB}" --demand_sample 20 \
  --split temporal --nneg_sweep --full_pool \
  --artifact_dir /content/drive/MyDrive/kipris_data/ipm_fp_temporal

## 셀 3 — 최종 random 10-seed + full_pool (~80분, **RQ1 대조**)
랜덤 분할(누수 비교군). temporal과의 대조가 RQ1. **끊기면 이 셀 그대로 다시.**

In [ ]:
!python run_ipm_experiment.py --mode full --device cuda \
  --data_dir "{DATA_DIR}" --emb_path "{EMB}" --demand_sample 200 \
  --split random --full_pool \
  --artifact_dir /content/drive/MyDrive/kipris_data/ipm_results_random

## 셀 4 — 결과 확인 (temporal + random)

In [ ]:
import os
for name, path in [('TEMPORAL (main)', '/content/drive/MyDrive/kipris_data/ipm_results_final/run_ipm_results.md'),
                   ('RANDOM (RQ1)',    '/content/drive/MyDrive/kipris_data/ipm_results_random/run_ipm_results.md')]:
    print('='*80); print(name, '->', path); print('='*80)
    print(open(path).read() if os.path.exists(path) else '(아직 없음 — 해당 셀 완료 후 생성됨)')
    print()

---
### 끊김 / 재개
- **죽어도 같은 셀을 그대로 다시 실행** → `_resume_checkpoint.pt`에서 죽은 시드부터 이어서(완료 시드 건너뜀, Demand-BFS 등 복원).
- 순서: GPU 확인 → **셀 1**(환경 복구) → 죽었던 셀 다시.
- 완전히 새로 시작하려면 **셀 1b**(초기화) 먼저, 또는 명령에 `--fresh_start` 추가.
- temporal·random은 폴더가 달라(ipm_results_final / ipm_results_random) **절대 안 섞임**.
- 탭 활성 유지 + 맥 안 자게(`caffeinate -d`). GPU 안 잡히면 무료 quota 소진(~12~24h 후 또는 다른 계정).